## Run both ansatz constructions locally


## Distinguish the hardware ansatz from the tapered solve

The visible four-bit qfunc composes explicit single-qubit rotations and a
three-layer entangler on the declared reference occupation. FlagQuantum lowers
that native composition. The host separately constructs the molecular fermion
Hamiltonian, mode ordering, tapered one-bit model and classical optimizer;
these are not all consequences of calling synth on a molecule.

Inspect four-wire resources separately from the actual tapered UCCSD solve.
The retained energy objective uses aggregate binomial counts equivalent to
one million shots per Pauli basis, rather than an exact expectation. A sampled
energy and convergence flag do not certify chemical accuracy or quantum
advantage. Propagate preview edits to workflow.py to alter its hardware ansatz.


In [ ]:
from flagquantum import synth
from flagquantum.synth import CX
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import R
from flagquantum.synth import X
from flagquantum.synth import allocate
from flagquantum.synth import qfunc
import math


@qfunc
def hardware_ansatz(angles: list[float], state: Output[QArray[QBit]]) -> None:
    allocate(4, state)
    X(state[0])
    X(state[2])
    for layer in range(3):
        for wire in range(4):
            X(state[wire])
            R(angles[4 * layer + wire], math.pi / 2, state[wire])
        for wire in range(3):
            CX(state[wire], state[wire + 1])


def build_hardware_preview(angles: tuple[float, ...]) -> synth.Program:
    if len(angles) != 12 or not all(map(math.isfinite, angles)):
        raise ValueError("The hardware-efficient ansatz requires twelve finite angles")
    return synth.synthesize(
        hardware_ansatz,
        classical_inputs={"angles": angles},
        max_qubits=4,
        max_workspace=0,
    )


from IPython.display import Image, display

from examples.synthesis.applications.chemistry.molecule_eigensolver.overview.main import (
    report,
    save_figure,
)
from examples.synthesis.applications.chemistry.molecule_eigensolver.overview.workflow import (
    GEOMETRIES,
    full_hamiltonian,
    run_workflow,
)


## Review the source geometries


In [ ]:
print(tuple(GEOMETRIES))


## Select the source H2 case at 0.735 angstrom


In [ ]:
print(GEOMETRIES["H2"])


## Generate the full four-mode Hamiltonian


In [ ]:
hamiltonian = full_hamiltonian()
print(
    [
        (term.coefficient, term.factors)
        for term in hamiltonian.terms
        if abs(term.coefficient) > 1e-12
    ]
)


## Synthesize three hardware-efficient layers


In [ ]:
result = run_workflow()
print("Four-wire resources:", result.hardware)


## Read the one-qubit paired UCCSD sector


In [ ]:
print("Identity, X and Z coefficients:", result.reduced)


## Read the finite-shot optimization result


In [ ]:
report(result)


## Review the optimized parameter


In [ ]:
print("Angle in radians:", result.angle)


## Compare the optimized state with full CI


In [ ]:
display(Image(filename=str(save_figure(result))))


preview = build_hardware_preview((0.01,) * 12)
print("Editable four-wire resources:", preview.report["resources"])
